# Notebook 03: CLI & Integration Tests

## Goal

This notebook tests the **command-line interface**, **cross-module data flow**, and
**edge cases** that only surface when the modules interact or receive unusual inputs.

Notebooks 01 and 02 verified that each module works correctly on its own and that the
inject-then-detect workflow produces the right results. This notebook asks different
questions:

1. **CLI** - Does the `ghost-in-the-template` command parse arguments correctly? Does it
   return the right exit codes for success, detection, and error conditions? Does it work
   when invoked as a subprocess (simulating real terminal usage)?

2. **Cross-module integration** - When the normalizer feeds into the integrity engine,
   do formatting-only diffs actually resolve? When the poisoning script feeds into the
   structural analyzer feeds into the reporter, does the full chain produce correct output?

3. **Edge cases** - What happens with very large templates (100 role branches)? Multiple
   backdoors in one template? Unicode characters? Templates that reference `message['content']`
   benignly (outputting it, not gating on it)? Does normalization preserve the AST structure?

### What we are trying to see

- Is the CLI a correct and complete interface to the detection pipeline?
- Do the modules compose correctly when data flows through the real call chain?
- Does the tool handle boundary conditions gracefully without crashing or producing
  incorrect results?
- Is the exception hierarchy correct (all custom errors inherit from GhostError)?

If every cell prints PASS, the tool is ready for real-world use from the command line.


---
## Setup

**What this cell does:** Adds the project root to `sys.path`, changes working directory
to the project root, and imports standard library modules used throughout the notebook.
`subprocess` is needed for CLI tests that invoke the tool as a separate process.
`json` is needed for parsing JSON output from the reporter.


In [ ]:
import sys, os, subprocess, json
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)
print(f'Working dir: {os.getcwd()}')


---
## 1. CLI Argument Parsing

The CLI (`cli.py`) uses argparse with three subcommands: `integrity`, `structural`,
and `extract`. Each subcommand accepts different arguments. We test that the parser
correctly handles valid inputs and rejects invalid ones.

These tests call `build_parser()` and `parse_args()` directly (in-process) rather than
spawning a subprocess. This is faster and lets us inspect the parsed `Namespace` object.
Subprocess tests come in Section 2.


**What this cell does:** Imports the CLI's `build_parser` and `main` functions, the
exit-code constants, and pytest (for `SystemExit` matching in later cells).


In [ ]:
from ghost_in_the_template.cli import build_parser, main
from ghost_in_the_template import EXIT_CLEAN, EXIT_DETECTION, EXIT_ERROR
import pytest


**What this cell does:** Tests five argument-parsing scenarios:

1. **`--version` flag** - Must exit with code 0 (argparse raises `SystemExit(0)` after
   printing the version). We catch the exception and verify the exit code.

2. **`integrity` subcommand** - Parses `integrity target.gguf --reference ref.gguf` and
   verifies `command`, `target`, `reference`, and `output_format` (default: 'text') are
   set correctly on the returned Namespace.

3. **`structural` subcommand** - Parses `structural target.gguf --format json` and
   verifies the format override is captured.

4. **`extract` subcommand** - Parses `extract file.gguf` and verifies the command name.

5. **Missing `--reference`** - `integrity target.gguf` without `--reference` must raise
   `SystemExit` because `--reference` is a required argument for the integrity subcommand.
   This tests argparse's built-in validation.


In [ ]:
parser = build_parser()

# Version flag
try:
    parser.parse_args(['--version'])
except SystemExit as e:
    assert e.code == 0
    print('PASS: --version exits with 0')

# Integrity command parsing
args = parser.parse_args(['integrity', 'target.gguf', '--reference', 'ref.gguf'])
assert args.command == 'integrity'
assert args.target == 'target.gguf'
assert args.reference == 'ref.gguf'
assert args.output_format == 'text'  # default
print('PASS: integrity command parses correctly')

# Structural command parsing
args = parser.parse_args(['structural', 'target.gguf', '--format', 'json'])
assert args.command == 'structural'
assert args.output_format == 'json'
print('PASS: structural command parses correctly')

# Extract command parsing
args = parser.parse_args(['extract', 'file.gguf'])
assert args.command == 'extract'
print('PASS: extract command parses correctly')

# Missing --reference should error
try:
    parser.parse_args(['integrity', 'target.gguf'])
    assert False
except SystemExit:
    print('PASS: integrity without --reference raises SystemExit')


---
## 2. CLI Error Handling

The tool follows a fail-closed design: any error must return `EXIT_ERROR` (code 2),
never `EXIT_CLEAN` (code 0). A silent success on error would be a security vulnerability,
as the user would believe a template has been verified when it hasn't.

We test error handling both in-process (calling `main()` directly) and via subprocess
(calling `python -m ghost_in_the_template.cli`, which simulates real terminal usage).


**What this cell does:** Tests four in-process error scenarios by calling `main()` with
different argument lists and checking the return value:

1. **No command** - `main([])` with no arguments. Must return EXIT_ERROR.
2. **Extract nonexistent file** - The file does not exist, so extraction must fail.
3. **Integrity with nonexistent files** - Both target and reference are missing.
4. **Structural with nonexistent file** - The target does not exist.

In all cases, `main()` must return EXIT_ERROR (2), not raise an unhandled exception.
The function catches exceptions internally and returns the appropriate exit code.


In [ ]:
# No command -> EXIT_ERROR
assert main([]) == EXIT_ERROR
print('PASS: No command returns EXIT_ERROR')

# Nonexistent file -> EXIT_ERROR
assert main(['extract', '/nonexistent/file.gguf']) == EXIT_ERROR
print('PASS: extract nonexistent file returns EXIT_ERROR')

assert main(['integrity', '/nonexistent/t.gguf', '-r', '/nonexistent/r.gguf']) == EXIT_ERROR
print('PASS: integrity nonexistent files returns EXIT_ERROR')

assert main(['structural', '/nonexistent/t.gguf']) == EXIT_ERROR
print('PASS: structural nonexistent file returns EXIT_ERROR')


### 2.1 CLI via subprocess (simulates real usage)

**What this cell does:** Invokes the CLI as a subprocess using `python -m ghost_in_the_template.cli`,
which is how a real user would run the tool from their terminal. This tests:

1. **Version output** - `--version` must exit 0 and print "0.1.0" to stdout.

2. **Help text** - `--help` must exit 0 and mention all three subcommands in the output.
   This confirms the argparse configuration is correct from the user's perspective.

3. **Error exit code** - Requesting extraction of a nonexistent file must return EXIT_ERROR
   (2) as the process exit code. This tests the fail-closed behavior end-to-end, including
   the `sys.exit()` call in `__main__.py` or the console_script entry point.


In [ ]:
# Version
r = subprocess.run(
    [sys.executable, '-m', 'ghost_in_the_template.cli', '--version'],
    capture_output=True, text=True
)
assert r.returncode == 0
assert '0.1.0' in r.stdout
print(f'PASS: CLI version output: {r.stdout.strip()}')

# Help
r = subprocess.run(
    [sys.executable, '-m', 'ghost_in_the_template.cli', '--help'],
    capture_output=True, text=True
)
assert r.returncode == 0
assert 'integrity' in r.stdout
assert 'structural' in r.stdout
assert 'extract' in r.stdout
print('PASS: CLI help shows all 3 subcommands')

# Nonexistent file via subprocess
r = subprocess.run(
    [sys.executable, '-m', 'ghost_in_the_template.cli', 'extract', '/no/such/file.gguf'],
    capture_output=True, text=True
)
assert r.returncode == EXIT_ERROR
print(f'PASS: extract nonexistent via subprocess returns {r.returncode}')


---
## 3. Cross-Module Integration

These tests verify that data flows correctly between modules. Unlike Notebook 01 (which
tested modules in isolation) and Notebook 02 (which tested the inject-then-detect workflow),
this section focuses on the **interfaces** between modules: does the output of one module
have the right shape and semantics to serve as input to the next?


### 3.1 Normalizer -> Integrity

**What this cell does:** Tests the critical integration point between the normalizer and
the integrity engine. Two templates that differ only in Jinja2 formatting are:

1. **Hashed raw** - Different hashes (the formatting differences change the bytes).
2. **Hashed after normalization** - Identical hashes (normalization resolved the differences).
3. **Checked via integrity engine** - `passed=True` (the engine normalizes internally).

This confirms that the integrity engine calls `normalize_template()` before hashing,
not after. If the normalization step were missing or applied to the wrong input,
formatting-only diffs would cause false FAIL results.


In [ ]:
from ghost_in_the_template.normalizer import normalize_template
from ghost_in_the_template.integrity import check_integrity_from_strings, compute_hash

# Templates that differ only in formatting
t1 = '{%  for m in msgs  %}{{  m  }}{%  endfor  %}\n'
t2 = '{% for m in msgs %}{{ m }}{% endfor %}\n'

# Raw hashes differ
assert compute_hash(t1) != compute_hash(t2)
print('Raw hashes differ (expected)')

# Normalized hashes match
assert compute_hash(normalize_template(t1)) == compute_hash(normalize_template(t2))
print('Normalized hashes match (expected)')

# Integrity check passes
r = check_integrity_from_strings(t1, t2)
assert r.passed
print('PASS: Normalizer -> Integrity integration works')


### 3.2 Poison -> Structural -> Reporter

**What this cell does:** Chains three modules in sequence:

1. `inject_payload()` produces a poisoned template from a clean one.
2. `analyze_template()` detects the content-gated-emit pattern.
3. `format_structural_result()` formats the finding as text and JSON.

We verify each handoff: the structural result is not clean, the text output contains
"SUSPICIOUS", and the JSON output parses correctly with `result: "SUSPICIOUS"`.

This tests a different data path than Notebook 02's E2E test. Here we construct a
minimal payload inline rather than reading from a file, isolating the module chain
from filesystem dependencies.


In [ ]:
from attack.poison import inject_payload
from ghost_in_the_template.structural import analyze_template
from ghost_in_the_template.reporter import format_structural_result

original = '{% for message in messages %}{{ message["content"] }}{% endfor %}\n'
payload = "{%- if message['role'] == 'user' and 'activate' in message['content'] %}\nInjected malicious instruction payload string.\n{%- endif %}"

poisoned = inject_payload(original, payload)
result = analyze_template(poisoned, 'test.gguf')

assert not result.clean
print('Structural detected the injection')

# Text output
txt = format_structural_result(result, 'text')
assert 'SUSPICIOUS' in txt
print(f'\nText report:\n{txt}')

# JSON output
j = json.loads(format_structural_result(result, 'json'))
assert j['result'] == 'SUSPICIOUS'
print('PASS: Full pipeline Poison -> Structural -> Reporter works')


### 3.3 Poison -> Integrity -> Reporter

**What this cell does:** Same chain concept but through the integrity path:

1. Uses the `poisoned` template from the previous cell.
2. `check_integrity_from_strings()` compares it against the clean `original`.
3. `format_integrity_result()` formats the FAIL result as text and JSON.

This verifies the integrity detection path produces correct reporter output when
given a real poisoned template (not a hand-crafted fixture).


In [ ]:
from ghost_in_the_template.reporter import format_integrity_result

r = check_integrity_from_strings(poisoned, original, 'poisoned.gguf', 'clean.gguf')
assert not r.passed

txt = format_integrity_result(r, 'text')
assert 'FAIL' in txt
print(f'Text report:\n{txt}')

j = json.loads(format_integrity_result(r, 'json'))
assert j['result'] == 'FAIL'
print('PASS: Full pipeline Poison -> Integrity -> Reporter works')


---
## 4. Edge Cases

Edge cases test behavior at the boundaries of expected input. These are the scenarios
most likely to cause crashes, false positives, or missed detections in production.
Each test targets a specific boundary condition.


### 4.1 Very Long Templates

**What this cell does:** Generates a template with 100 role branches (300+ lines) and
runs the structural analyzer on it. The purpose is threefold:

1. **No crash** - The AST walker must handle large node trees without stack overflow
   or excessive memory use.

2. **No false positive** - All 100 branches test `message['role']` (a structural field),
   not `message['content']`. None should be flagged.

3. **Performance sanity** - If this cell takes more than a few seconds, the analyzer
   has a performance problem (100 If nodes is modest by real-world standards).


In [ ]:
# Generate a large benign template
long_template = '{%- for message in messages %}\n'
for i in range(100):
    long_template += f'{{% if message["role"] == "role_{i}" %}}\n'
    long_template += f'<|role_{i}|>{{{{ message["content"] }}}}</s>\n'
    long_template += '{% endif %}\n'
long_template += '{%- endfor %}\n'

r = analyze_template(long_template)
assert r.clean
print(f'PASS: Large template ({len(long_template)} chars, 100 role branches) is clean')


### 4.2 Multiple Backdoors in One Template

**What this cell does:** Tests a template with two separate content-gated-emit blocks,
each with a different trigger phrase. The analyzer must find both.

This catches a bug where the analyzer might short-circuit after the first finding and
miss subsequent ones. Since `ast.find_all(nodes.If)` returns all If nodes, and each
is checked independently, both should be reported. We verify `len(findings) >= 2`
and print each finding's trigger expression.


In [ ]:
multi_backdoor = '''
{%- for message in messages %}
{%- if 'trigger_alpha' in message['content'] %}
Payload alpha injected instruction text here.
{%- endif %}
{%- if 'trigger_beta' in message['content'] %}
Payload beta injected instruction text here.
{%- endif %}
{%- endfor %}
'''

r = analyze_template(multi_backdoor)
assert not r.clean
assert len(r.findings) >= 2
print(f'PASS: {len(r.findings)} findings detected in multi-backdoor template')
for i, f in enumerate(r.findings):
    print(f'  Finding {i+1}: {f.trigger_expression}')


### 4.3 Unicode in Templates

**What this cell does:** Tests that templates containing non-ASCII Unicode characters
(CJK, emoji, accented characters, etc.) are handled correctly by both the structural
analyzer and the normalizer.

Real-world chat templates often contain Unicode for multilingual support. The tool must
not crash on non-ASCII input, and normalization must preserve the characters.


In [ ]:
unicode_template = '''
{%- for message in messages %}
{%- if message['role'] == 'user' %}
<|user|>{{ message['content'] }}</s>
{%- endif %}
{%- endfor %}
'''

r = analyze_template(unicode_template)
assert r.clean
print('PASS: Template with unicode characters is clean')

# Normalize unicode template
from ghost_in_the_template.normalizer import normalize_template
norm = normalize_template(unicode_template)
assert norm  # non-empty
print('PASS: Unicode template normalizes without error')


### 4.4 Template with Only Whitespace Differences from Backdoor Perspective

**What this cell does:** Tests a template that references `message['content']` inside
an If body, but only to **output** it (via `{{ message['content'] }}`), not to **gate**
on it (via `'trigger' in message['content']`).

This is a critical false-positive test. The pattern `if role == 'user': output content`
is extremely common in benign templates. The analyzer must distinguish between:

- **Gating on content** (the If test checks content): suspicious
- **Outputting content** (the If body emits content): benign

The difference is in the AST: gating means the Compare/In node references content,
while outputting means an Output node inside the body does. Only the former is flagged.


In [ ]:
# A template that references content but in a benign way
# (outputting it, not gating on it)
benign_content_use = '''
{%- for message in messages %}
{%- if message['role'] == 'user' %}
User said: {{ message['content'] }}
{%- endif %}
{%- endfor %}
'''

r = analyze_template(benign_content_use)
assert r.clean, 'Benign content output should not be flagged'
print('PASS: Benign content output not flagged as suspicious')


### 4.5 Exception Hierarchy

**What this cell does:** Verifies the custom exception class hierarchy defined in
`__init__.py`:

```
Exception
  +-- GhostError (base for all tool errors)
        +-- ExtractionError (GGUF parsing/reading failures)
        +-- NormalizationError (invalid input to normalizer)
        +-- AnalysisError (Jinja2 parse failures in structural analyzer)
```

This hierarchy lets callers catch `GhostError` to handle any tool error, or catch
specific subclasses for targeted handling. The CLI uses this: it catches `GhostError`
at the top level to produce a formatted error report and return EXIT_ERROR.


In [ ]:
from ghost_in_the_template import GhostError, ExtractionError, NormalizationError, AnalysisError

# All custom exceptions inherit from GhostError
assert issubclass(ExtractionError, GhostError)
assert issubclass(NormalizationError, GhostError)
assert issubclass(AnalysisError, GhostError)

# GhostError inherits from Exception
assert issubclass(GhostError, Exception)

print('PASS: Exception hierarchy correct')
print(f'  GhostError -> Exception')
print(f'  ExtractionError -> GhostError')
print(f'  NormalizationError -> GhostError')
print(f'  AnalysisError -> GhostError')


### 4.6 Normalization Preserves Logic

**What this cell does:** The most important correctness property of the normalizer:
normalization must not alter the template's logical structure.

We verify this by parsing a template before and after normalization, then comparing
the AST node counts. If normalization added, removed, or changed any AST node, the
counts would differ.

The test template has deliberately messy formatting (`{%   for`, `{%  if`, `{{   }}`,
etc.) to exercise every normalization step. After normalization, the formatting is
canonical, but the AST structure (For, If, Output, TemplateData nodes) must be identical.

A failure here would mean normalization is too aggressive: it is changing template
semantics, not just formatting. That would undermine the integrity check (templates
that are logically different could hash the same after normalization).


In [ ]:
# Verify that normalization does NOT alter template logic
# by checking that a normalized template still parses to the same AST structure
import jinja2

template = '''{%   for message   in   messages   %}
{%  if   message['role']   ==   'user'  %}
<|user|>{{   message['content']   }}</s>
{%   endif   %}
{%  endfor  %}\n'''

normalized = normalize_template(template)

env = jinja2.Environment()
ast_original = env.parse(template)
ast_normalized = env.parse(normalized)

# Count nodes in both ASTs
def count_nodes(ast):
    count = 0
    for node in ast.find_all(jinja2.nodes.Node):
        count += 1
    return count

orig_count = count_nodes(ast_original)
norm_count = count_nodes(ast_normalized)

assert orig_count == norm_count, f'AST node count changed: {orig_count} -> {norm_count}'
print(f'PASS: Normalization preserves AST structure ({orig_count} nodes)')


---
## All CLI & Integration Tests Passed

The tool is ready for real-world command-line use:
- The CLI correctly parses all subcommands and returns appropriate exit codes.
- Modules compose correctly with data flowing through the real call chain.
- Edge cases (large templates, multiple backdoors, unicode, benign content references)
  are handled gracefully with no crashes or false positives.
- The exception hierarchy is correct and the normalizer preserves template logic.
